# Public Transport Delay Prediction using Machine Learning

## MLflow Experiment Tracking

### Objective

Track the completed modelling experiments from Notebooks 3, 4, and 5 using **MLflow**.

This notebook focuses on experiment tracking rather than retraining the models. The results already produced in the modelling notebooks are logged with:

- model names
- hyperparameters
- validation metrics
- test metrics where available
- model artefacts
- evaluation result files
- explainability artefacts

### Why MLflow?

The modelling notebooks answer:

> **How well does each model perform?**

MLflow adds:

> **Which experiment produced that result, with which parameters and artefacts?**

This creates a reproducible experiment history that can be inspected through the MLflow Tracking UI.

> This notebook tracks the experiments already completed. It does not create new model performance claims by retraining the models.


## Project Tracking Strategy

The project contains three modelling tasks:

| Task | Source | What will be tracked |
|---|---|---|
| Classification | Notebook 3 | Cross-validation results and Random Forest diagnostic test metrics |
| Regression | Notebook 4 | Candidate-model CV results, tuned Random Forest parameters, final test metrics, saved model |
| Time Series | Notebook 5 | Validation results, selected 3-Day Moving Average, final test metrics, forecast artefacts |
| Explainability | Notebook 6 | Regression explainability CSVs and figures attached to the final regression run |

The classification run is explicitly tagged as a **diagnostic** because Notebook 3 identified a target-construction limitation.


In [1]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

print("MLflow imported successfully.")
print("MLflow version:", mlflow.__version__)

MLflow imported successfully.
MLflow version: 3.16.1


## Configure Local MLflow Tracking

A local SQLite database is used to store MLflow tracking metadata for this project.

The database is placed at the project root as:

`mlflow.db`

The approach keeps experiment metadata local and reproducible while avoiding the need for an external tracking server during development.

In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

# SQLite tracking URI. The absolute path keeps the location stable
# regardless of the notebook's current working directory.
TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"

mlflow.set_tracking_uri(TRACKING_URI)

EXPERIMENT_NAME = "Public Transport Delay Prediction"

mlflow.set_experiment(EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)


2026/09/27 22:12:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/27 22:12:46 INFO mlflow.store.db.utils: Updating database tables
2026/09/27 22:12:53 INFO mlflow.tracking.fluent: Experiment with name 'Public Transport Delay Prediction' does not exist. Creating a new experiment.


Tracking URI: sqlite:///C:/Users/srisa/OneDrive/Desktop/Projects/Public Transport Delays/mlflow.db
Experiment: Public Transport Delay Prediction
Experiment ID: 1


## Define Project Paths

The paths below match the artefact structure used by the completed notebooks.

- Saved regression model → `artifacts/models/`
- Saved time-series model → `artifacts/models/`
- Regression results → `results/regression/`
- Time-series results → `results/time_series/`
- Explainability results → `results/explainability/`
- MLflow database → project root

In [3]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"

REGRESSION_RESULTS_DIR = PROJECT_ROOT / "results" / "regression"
TIME_SERIES_RESULTS_DIR = PROJECT_ROOT / "results" / "time_series"
EXPLAINABILITY_RESULTS_DIR = PROJECT_ROOT / "results" / "explainability"
MLFLOW_RESULTS_DIR = PROJECT_ROOT / "results" / "mlflow"

MLFLOW_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

regression_model_path = MODELS_DIR / "best_regression_pipeline.joblib"
time_series_model_path = MODELS_DIR / "best_time_series_model.joblib"

print("Project root:", PROJECT_ROOT)
print("Regression model exists:", regression_model_path.exists())
print("Time-series model exists:", time_series_model_path.exists())
print("Explainability results directory exists:", EXPLAINABILITY_RESULTS_DIR.exists())


Project root: C:\Users\srisa\OneDrive\Desktop\Projects\Public Transport Delays
Regression model exists: True
Time-series model exists: True
Explainability results directory exists: True


## Helper Functions

The functions below keep the logging cells readable.

Each MLflow run records parameters, metrics, tags, and optional artefacts.

The regression model is also logged through MLflow's scikit-learn model integration when available, while the original `.joblib` file is retained as an additional artefact.


In [4]:
import numpy as np
import pandas as pd
import joblib

def start_logged_run(run_name, task, source_notebook, tags=None):
    base_tags = {
        "project": "public_transport_delay_prediction",
        "task": task,
        "source_notebook": source_notebook,
        "tracking_type": "completed_experiment"
    }

    if tags:
        base_tags.update(tags)

    return mlflow.start_run(
        run_name=run_name,
        tags=base_tags
    )


def log_metric_dict(metrics):
    for name, value in metrics.items():
        if value is not None and pd.notna(value):
            mlflow.log_metric(name, float(value))


def log_existing_file(path, artifact_path=None):
    path = Path(path)

    if path.exists():
        mlflow.log_artifact(
            str(path),
            artifact_path=artifact_path
        )
        return True

    print("Skipped missing artefact:", path)
    return False


def log_sklearn_model_compatible(model, name, signature=None, input_example=None):
    # MLflow's current API uses the `name` argument. A fallback keeps
    # the notebook usable with older MLflow releases.
    try:
        return mlflow.sklearn.log_model(
            model,
            name=name,
            signature=signature,
            input_example=input_example
        )
    except TypeError:
        return mlflow.sklearn.log_model(
            model,
            artifact_path=name,
            signature=signature,
            input_example=input_example
        )

print("MLflow helper functions created.")


MLflow helper functions created.


# 1. Classification Experiment Tracking

Notebook 3 evaluated multiple classifiers and found that the strict pre-trip classification problem has a dataset-level limitation: the binary `delayed` target exactly matches whether `actual_arrival_delay_min > 5`.

The Random Forest diagnostic therefore should be tracked as a **diagnostic experiment**, not presented as a validated high-performing production classifier.